# EDA to understand the data

In [1]:
# Import libraries
import pandas as pd
from pathlib import Path

## View basic info of each raw data file

### Preliminary findings
- Data is relatively clean, very few non-null. 
- Tables with null values include reviews and products table
    - Not all customers leave reviews
    - There are products with product_id but no product name

In [2]:
for file in Path("../data").iterdir():
    if file.is_file():
        df = pd.read_csv(file)
        initial_shape = df.shape
        print(f"{'-'*20} Read datafile {file} with shape: {initial_shape} {'-'*20}")
        df.info()
        df.drop_duplicates(inplace=True)
        if df.shape[0] < initial_shape[0]:
            print(f"{initial_shape[0] - df.shape[0]} duplicated rows found")
        dt_cols = []
        for col in df.columns:
            if "_id" in col:
                # Check if each row is unique and each row is not null
                if df[col].nunique() != df.shape[0]:
                    print(f"Column {col} is Non-unique despite being an identifier.")
                if df[col].isna().any():
                    print(f"Column {col} contains Null values")
            if "date" in col or "timestamp" in col:
                try:
                    df[col] = pd.to_datetime(df[col])
                    dt_cols.append(col)
                except Exception as e:
                    print(f"Error encountered in converting {col} to datetime: {e}")
        if dt_cols:
            print(f"Datetime columns: {dt_cols}")
        else:
            print("No datetime columns found.")
        print()

-------------------- Read datafile ../data/olist_sellers_dataset.csv with shape: (3095, 4) --------------------
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB
No datetime columns found.

-------------------- Read datafile ../data/product_category_name_translation.csv with shape: (71, 2) --------------------
<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_nam

## Deeper dive to each dataset

### Customers data

In [3]:
customers_df = pd.read_csv("../data/olist_customers_dataset.csv")
print(customers_df.shape)
customers_df.head(2)

(99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [4]:
print(customers_df["customer_id"].nunique(), customers_df["customer_unique_id"].nunique())
print("Mismatched customer id unique count. This is because sip code of customer_id may change, hence customer_unique_id is the unique identifier whereas customer_id is tagged to an order")

99441 96096
Mismatched customer id unique count. This is because sip code of customer_id may change, hence customer_unique_id is the unique identifier whereas customer_id is tagged to an order


In [5]:
customers_minimal_df = customers_df.drop(columns="customer_id")
customers_minimal_df = customers_minimal_df.drop_duplicates()
print(customers_minimal_df.shape)

(96352, 4)


In [6]:
# Sample customers with duplicated unique_id due to difference in customer_zip_code_prefix
customers_minimal_df[customers_minimal_df.duplicated(subset='customer_unique_id', keep=False)].sort_values(by="customer_unique_id").tail(6)

,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
69820,fd09c64a101e3eff4adbca1b28552514,8542,ferraz de vasconcelos,SP
84770,fd09c64a101e3eff4adbca1b28552514,8070,sao paulo,SP
76055,fe3e52de024b82706717c38c8e183084,72306,brasilia,DF
59713,fe3e52de024b82706717c38c8e183084,36420,ouro branco,MG
78212,fe59d5878cd80080edbd29b5a0a4e1cf,71065,brasilia,DF
23933,fe59d5878cd80080edbd29b5a0a4e1cf,71065,guara,DF


### Reviews dataframe

In [7]:
reviews_df = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# View reviews with non NA title. Notice the language is in Spanish
reviews_df[reviews_df["review_comment_title"].notna()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01
19,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo,2018-08-15 00:00:00,2018-08-15 04:10:37
22,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10,2018-07-10 00:00:00,2018-07-11 14:10:25
34,c92cdd7dd544a01aa35137f901669cdf,37e7875cdce5a9e5b3a692971f370151,4,Muito bom.,Recebi exatamente o que esperava. As demais en...,2018-06-07 00:00:00,2018-06-09 18:44:02
...,...,...,...,...,...,...,...
99192,0e7bc73fde6782891898ea71443f9904,bd78f91afbb1ecbc6124974c5e813043,4,👍,Aprovado!,2018-07-04 00:00:00,2018-07-05 00:25:13
99196,58be140ccdc12e8908ff7fd2ba5c7cb0,0ebf8e35b9807ee2d717922d5663ccdb,5,muito bom produto,"Ficamos muito satisfeitos com o produto, atend...",2018-06-30 00:00:00,2018-07-02 23:09:35
99197,51de4e06a6b701cb2be47ea0e689437b,b7467ae483dbe956fe9acdf0b1e6e3f4,3,Não foi entregue o pedido,Bom dia \r\nDas 6 unidades compradas só recebi...,2018-06-05 00:00:00,2018-06-06 10:52:19
99199,40743b46a0ee86375cedb95e82b78d75,3e93213bb8fdda91186b4018b2fe0030,5,OTIMA EMBALAGEM,NaN,2018-08-08 00:00:00,2018-08-08 16:56:16


In [8]:
print("Notice the review_id column become unique after dropping order_id column")
print("Unsure why different orders are being tagged to the same review")
print(reviews_df.drop(columns="order_id").drop_duplicates()["review_id"].nunique())
reviews_df.drop(columns="order_id").drop_duplicates().sort_values(by="review_id")

Notice the review_id column become unique after dropping order_id column
Unsure why different orders are being tagged to the same review
98410


,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
56960,0001239bc1de2e33cb583967c2ca4c67,5,NaN,NaN,2018-03-20 00:00:00,2018-03-20 18:36:04
20532,0001cc6860aeaf5b9017fe4131a52e62,5,NaN,NaN,2018-06-22 00:00:00,2018-06-26 13:51:29
50129,00020c7512a52e92212f12d3e37513c0,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...,2018-04-25 00:00:00,2018-04-26 14:55:36
54237,00032b0141443497c898b3093690af51,5,NaN,NaN,2017-05-30 00:00:00,2017-06-01 23:28:55
59910,00034d88989f9a4c393bdcaec301537f,5,NaN,NaN,2017-08-12 00:00:00,2017-08-13 19:56:53
...,...,...,...,...,...,...
31591,fffcfa6087cd3b651c68252342f13cb9,4,NaN,NaN,2017-07-14 00:00:00,2017-07-15 02:14:44
61626,fffd24e2cf1ca4ee917e2f05be3c01fb,5,NaN,NaN,2017-11-25 00:00:00,2017-11-25 21:23:00
88923,fffd68e8a9fb73a56a2f504011b0f1f1,1,NaN,Produto muito ruim.,2017-05-31 00:00:00,2017-06-02 23:21:51
10798,fffee432d53abd67b5b0fd4fc290d8c3,5,NaN,NaN,2017-10-25 00:00:00,2017-10-26 08:17:24


In [9]:
# View reviews that have the same review_id
reviews_df[reviews_df.duplicated(subset='review_id', keep=False)&
           reviews_df["review_comment_message"].notna()].sort_values(by="review_id").head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
9013,03a6a25db577d0689440933055111897,3fde8b7313af6b37b84b5c7594d7add0,5,NaN,Muito Bom! Gostei Bastante! Tecido Ótimo! Aten...,2017-12-15 00:00:00,2017-12-16 01:32:18
1988,03a6a25db577d0689440933055111897,2acfdc5131ff2cf4433e668454c9784c,5,NaN,Muito Bom! Gostei Bastante! Tecido Ótimo! Aten...,2017-12-15 00:00:00,2017-12-16 01:32:18
77444,0498e9722f8757426c3c3ee0b91e666d,041aa5c38550649d5b51f38ba03a29a4,4,NaN,Chego certo adoro comprar na lannister.com.br\...,2018-03-03 00:00:00,2018-03-06 01:00:33
96681,0498e9722f8757426c3c3ee0b91e666d,f318811b0fd898d1edf78d6841470be2,4,NaN,Chego certo adoro comprar na lannister.com.br\...,2018-03-03 00:00:00,2018-03-06 01:00:33


### Orders

`order_status` column and how the table gets populated:
1. `created` (order_purchase_timestamp and order_estimated_delivery_date populated) -> 
2. `approved` (order_approved_at populated) ->
3. `processing` / `invoiced` / `unavailable` (order_id is populated in payments_df) -> 
4. `shipped` (order_delivered_carrier_date populated) ->
5. `delivered` (order_delivered_customer_date populated)

**`canceled` can happen at any stage (before approval, after carrier has delivered but before customer receives etc)

In [10]:
orders_df = pd.read_csv("../data/olist_orders_dataset.csv")
print(orders_df.shape)
orders_df["order_status"].value_counts()

(99441, 8)


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [11]:
orders_df[orders_df["order_status"].isin(["delivered"])] 

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [12]:
order_items_df = pd.read_csv("../data/olist_order_items_dataset.csv")
print(order_items_df.shape)
order_items_df.head(2)

(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


In [13]:
print("Number of items in one order can go up to 21")
order_items_df.describe()

Number of items in one order can go up to 21


,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [14]:
## Looking at the order with 21 items
order_items_df[order_items_df["order_id"] == (order_items_df.loc[order_items_df["order_item_id"]==21, "order_id"].values[0])]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [15]:
# Check that (order_id, order_item_id) is unique and can be a primary key
order_items_df[["order_id", "order_item_id"]].value_counts().sort_values()

order_id                          order_item_id
00010242fe8c5a6d1ba2dd792cb16214  1                1
ab14fdcfbe524636d65ee38360e22ce8  3                1
                                  2                1
                                  1                1
ab1430e26162f925a945d18941057aee  1                1
                                                  ..
5539bd029cf95d97ba8f51f6f323c839  1                1
553901a853048dcd33ec8de19f90c5d0  1                1
5536d8682e25ee3f39ed444105996fee  1                1
553c468f68af19869694de8bcd095213  2                1
fffe41c64501cc87c801fd61db3f6244  1                1
Name: count, Length: 112650, dtype: int64

In [16]:
# Check that each product is not always tagged to the same seller
product_seller_df = order_items_df[["product_id", "seller_id"]].drop_duplicates()
print(product_seller_df.shape)
print(product_seller_df["product_id"].nunique())
print(product_seller_df["seller_id"].nunique())
print("There are 34448 product-seller combination. With 32951 unique products. Meaning there are duplicated products for different sellers")
product_seller_df[product_seller_df.duplicated(subset="product_id",keep=False)].sort_values(by="product_id")

(34448, 2)
32951
3095
There are 34448 product-seller combination. With 32951 unique products. Meaning there are duplicated products for different sellers


,product_id,seller_id
43673,00f179926ae5965cd6ff548d765d8a61,f8db351d8c4c4c22c6835c19a46f01b0
11412,00f179926ae5965cd6ff548d765d8a61,8b28d096634035667e8263d57ba3368c
12130,00f8c37377b038c9c791128d2f928111,dbd66278cbfe1aa1000f90a217ca4695
108309,00f8c37377b038c9c791128d2f928111,5acd070dd3fe441bbb2ec1f1ede515ee
42150,00fefaf41156bb4b0d850fb27da97897,e9779976487b77c6d4ac45f75ec7afe9
...,...,...
43975,ffd2365fb8224dc66883df9351d65deb,fa1c13f2614d7b5c4749cbc52fecda94
50555,ffd2365fb8224dc66883df9351d65deb,2bf6a2c1e71bbd29a4ad64e6d3c3629f
40667,ffd2365fb8224dc66883df9351d65deb,606ce7768feac12c5e8bd58db8b08f0f
1032,ffd34459c21034d1da6df9800de0d7a3,11bfa66332777660bd0640ee84d47006


In [17]:
# Check that each order_id exists in both order_items and orders_df, since inner join maintains the same number of rows as the larger table
print(order_items_df.shape, orders_df.shape)
orders_items_joined_df = order_items_df.merge(orders_df, on="order_id", how='inner')
print(orders_items_joined_df.shape)
orders_items_joined_df.head(2)

(112650, 7) (99441, 8)
(112650, 14)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00


### Payments dataset

In [18]:
payments_df = pd.read_csv("../data/olist_order_payments_dataset.csv")
print("Note that payment_sequential can go up to 29")
payments_df.describe()

Note that payment_sequential can go up to 29


,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [19]:
# View orders with more than one order_id, and how the payment value varies by payment_sequential
payments_df[payments_df.duplicated(subset='order_id', keep=False)].sort_values(by=["order_id", "payment_sequential"]).head(20)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
16053,009ac365164f8e06f59d18a08045f6c4,1,credit_card,1,0.88
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
73837,009ac365164f8e06f59d18a08045f6c4,3,voucher,1,8.25
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45


In [20]:
# Testing if orders with `processing` status appear in payments_df.
# This can be tested with other order_status values.
payments_df[payments_df["order_id"].isin(orders_df[orders_df["order_status"].isin(["processing"])]["order_id"].to_list())]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
193,c79bdf061e22288609201ec60deb42fb,1,credit_card,1,12.22
413,ede510a059fd07ccec20bff19a227fa3,1,boleto,1,203.16
753,6c98fa39891b33399785aeac3d7ee926,1,credit_card,2,424.04
792,055f84fd2ebf314ebec85f8e1312f4f4,1,credit_card,2,185.84
1549,ebdc1743a498b33e5f50d4270c117175,1,credit_card,4,142.81
...,...,...,...,...,...
102759,79ec7fcffa4121dbb55b945aee306e12,1,boleto,1,249.56
103104,43bd50a07cc1758b29b32804f1a6d552,1,credit_card,1,57.67
103258,4bcb31b6a61c7fdbcb397a502edd29a1,1,credit_card,4,44.10
103281,ca7d7c95b0f170aae8a2ee59e2f24f66,1,voucher,1,20.00


### Sellers dataset

In [21]:
sellers_df = pd.read_csv("../data/olist_sellers_dataset.csv")
print(sellers_df.shape, sellers_df["seller_id"].nunique())
print("Note that seller_id is unique")
sellers_df.head(2)

(3095, 4) 3095
Note that seller_id is unique


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


### Product dataset

In [22]:
product_df = pd.read_csv("../data/olist_products_dataset.csv")
print(product_df.shape, product_df["product_id"].nunique())
print("Note that product_id is unique")
product_df.head(2)

(32951, 9) 32951
Note that product_id is unique


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


### Product Category Name dataset

In [23]:
product_names_df = pd.read_csv("../data/product_category_name_translation.csv")
print(product_names_df.shape)
# Check that the Spanish to English name is a 1-1 mapping
assert product_names_df["product_category_name"].nunique() == product_names_df["product_category_name_english"].nunique()
product_names_df.head(20)

(71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


### Geolocation dataset

In [24]:
geolocation_df = pd.read_csv("../data/olist_geolocation_dataset.csv")
print(geolocation_df.shape, geolocation_df["geolocation_zip_code_prefix"].nunique())
print("Note that there are many duplicates of geolocation_zip_code_prefix")

(1000163, 5) 19015
Note that there are many duplicates of geolocation_zip_code_prefix


In [25]:
# Note that geolocation_state is a two-letter capitalised value
geolocation_df["geolocation_state"].value_counts()

geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64

In [26]:
# Drop duplicates accounting only for zip code prefix, city and state
prefix_city_state_df = geolocation_df.drop_duplicates(subset=["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"], keep="first")
print(prefix_city_state_df.shape)
print("After dropping columns geolocation_lat and geolocation_lng, there are still duplicated rows due to Spanish encoding issues for geolocation_city column")
prefix_city_state_df[prefix_city_state_df.duplicated(subset="geolocation_zip_code_prefix", keep=False)].sort_values(by="geolocation_zip_code_prefix")

(27912, 5)
After dropping columns geolocation_lat and geolocation_lng, there are still duplicated rows due to Spanish encoding issues for geolocation_city column


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
99,1001,-23.549292,-46.633559,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
73,1002,-23.548318,-46.635421,sao paulo,SP
1251,1002,-23.544641,-46.633180,são paulo,SP
478,1003,-23.549083,-46.634864,são paulo,SP
...,...,...,...,...,...
999744,99940,-28.056947,-51.856821,ibiaca,RS
999774,99955,-28.107588,-52.144019,vila langaro,RS
1000046,99955,-28.107588,-52.144019,vila lângaro,RS
999780,99970,-28.345143,-51.876926,ciriaco,RS
